# Bluestock Fintech - Mutual Fund Analytics Capstone
## Day 4: Fund Performance & Risk Analytics
**Author:** Rajeev Reddy


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy import stats
from pathlib import Path
import os

# Project base path
BASE_DIR = Path(r"C:\blue\drive-download-20260902T170546Z-1-001")
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR = BASE_DIR

print("=" * 60)
print("BLUESTOCK FINTECH - CAPSTONE PROJECT")
print("Day 4: Fund Performance Analytics")
print("=" * 60)

### Load data

In [ ]:
print("\n[Loading data...")
nav_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "clean_nav.csv"))
perf_clean = pd.read_csv(os.path.join(PROCESSED_DIR, "clean_performance.csv"))
fund_master = pd.read_csv(os.path.join(RAW_DIR, "01_fund_master.csv"))
bench_raw = pd.read_csv(os.path.join(RAW_DIR, "10_benchmark_indices.csv"))

nav_clean["date"] = pd.to_datetime(nav_clean["date"])
bench_raw["date"] = pd.to_datetime(bench_raw["date"])

print("  NAV history: " + str(nav_clean.shape[0]) + " rows")
print("  Performance data: " + str(perf_clean.shape[0]) + " rows")

### Task 1: Compute daily returns for all funds

In [ ]:
print("\n[Task 1] Computing daily returns for all funds...")

# Calculate daily returns: nav_t / nav_t-1 - 1
daily_returns = []
for code in nav_clean["amfi_code"].unique():
    fund_nav = nav_clean[nav_clean["amfi_code"] == code].sort_values("date")
    fund_nav["daily_return"] = fund_nav["nav"].pct_change()
    daily_returns.extend(fund_nav[["date", "amfi_code", "daily_return"]].to_dict("records"))

dr_df = pd.DataFrame(daily_returns)
dr_df.to_csv(os.path.join(PROCESSED_DIR, "daily_returns.csv"), index=False)
print("  [OK] Daily returns computed and saved: " + str(dr_df.shape[0]) + " records")

### Task 2: Calculate CAGR for 1yr, 3yr, 5yr periods

In [ ]:
print("\n[Task 2] Calculating CAGR for 1yr, 3yr, 5yr periods...")

cagr_results = []
for code in perf_clean["amfi_code"].unique():
    fund_perf = perf_clean[perf_clean["amfi_code"] == code].iloc[0]
    return_1yr = fund_perf["return_1yr_pct"]
    return_3yr = fund_perf["return_3yr_pct"]
    return_5yr = fund_perf["return_5yr_pct"]
    
    cagr_results.append({
        "amfi_code": code,
        "return_1yr_pct": return_1yr,
        "return_3yr_cagr": return_3yr,
        "return_5yr_cagr": return_5yr
    })

cagr_df = pd.DataFrame(cagr_results)
cagr_df.to_csv(os.path.join(PROCESSED_DIR, "cagr_report.csv"), index=False)
print("  [OK] CAGR report saved: " + str(cagr_df.shape[0]) + " funds")

### Task 3: Compute Sharpe Ratio

In [ ]:
print("\n[Task 3] Computing Sharpe Ratio...")

# Sharpe = (Rp - Rf) / Std(Rp)  Use Rf = 6.5% (RBI repo rate proxy)
# Annualise with sqrt(252)
Rf_daily = 0.065 / 252  # Daily risk-free rate

sharpe_results = []
for code in perf_clean["amfi_code"].unique():
    fund_returns = dr_df[dr_df["amfi_code"] == code]["daily_return"].dropna()
    if len(fund_returns) > 1 and fund_returns.std() > 0:
        sharpe = (fund_returns.mean() - Rf_daily) / fund_returns.std() * np.sqrt(252)
    else:
        sharpe = np.nan
    sharpe_results.append({"amfi_code": code, "sharpe_ratio": sharpe})

sharpe_df = pd.DataFrame(sharpe_results)
sharpe_df.to_csv(os.path.join(PROCESSED_DIR, "sharpe_report.csv"), index=False)
print("  [OK] Sharpe ratio report saved: " + str(sharpe_df.shape[0]) + " funds")
print("  Note: Risk-free rate (Rf) = 6.5% (RBI repo rate proxy)")

### Task 4: Compute Sortino Ratio

In [ ]:
print("\n[Task 4] Computing Sortino Ratio...")

# Sortino = (Rp - Rf) / Downside_Std where Downside_Std uses only negative return days
sortino_results = []
for code in perf_clean["amfi_code"].unique():
    fund_returns = dr_df[dr_df["amfi_code"] == code]["daily_return"].dropna()
    if len(fund_returns) > 1:
        downside_returns = fund_returns[fund_returns < 0]
        downside_std = downside_returns.std() if len(downside_returns) > 0 else 0
        if downside_std > 0:
            sortino = (fund_returns.mean() - Rf_daily) / downside_std * np.sqrt(252)
        else:
            sortino = np.nan
    else:
        sortino = np.nan
    sortino_results.append({"amfi_code": code, "sortino_ratio": sortino})

sortino_df = pd.DataFrame(sortino_results)
sortino_df.to_csv(os.path.join(PROCESSED_DIR, "sortino_report.csv"), index=False)
print("  [OK] Sortino ratio report saved: " + str(sortino_df.shape[0]) + " funds")

### Task 5: Alpha & Beta from original performance data

In [ ]:
print("\n[Task 5] Alpha & Beta from original performance data...")

# The clean_performance.csv already has alpha and beta columns from original AMFI data
print("  Alpha NaN count in clean_performance: " + str(perf_clean["alpha"].isna().sum()))
print("  Beta NaN count in clean_performance: " + str(perf_clean["beta"].isna().sum()))
print("  Using original Alpha & Beta values from AMFI performance data")

### Task 6: Compute Maximum Drawdown

In [ ]:
print("\n[Task 6] Computing Maximum Drawdown...")

max_dd_results = []
for code in perf_clean["amfi_code"].unique():
    fund_nav = nav_clean[nav_clean["amfi_code"] == code].sort_values("date")
    # Running maximum
    running_max = fund_nav["nav"].cummax()
    # Drawdown = (NAV / running_max) - 1 (negative values indicate drawdown)
    dd = (fund_nav["nav"] / running_max - 1).min()
    max_dd_results.append({
        "amfi_code": code,
        "max_drawdown_pct": dd
    })

max_dd_df = pd.DataFrame(max_dd_results)
max_dd_df.to_csv(os.path.join(PROCESSED_DIR, "max_drawdown_report.csv"), index=False)
print("  [OK] Max drawdown report saved: " + str(max_dd_df.shape[0]) + " funds")

### Task 7: Build Fund Scorecard (composite score 0-100)

In [ ]:
print("\n[Task 7] Building Fund Scorecard (composite score 0-100)...")

# Merge performance data with fund master for scheme names and fund houses
merged = perf_clean.merge(fund_master[["amfi_code", "scheme_name", "fund_house"]], on="amfi_code")

# Check actual column names after merge
print("  Merged columns:", [c for c in merged.columns if 'scheme' in c.lower() or 'fund' in c.lower() or 'house' in c.lower()])

# Use the scheme_name and fund_house columns that exist
scheme_col = [c for c in merged.columns if c == "scheme_name"]
if len(scheme_col) == 0:
    scheme_col = [c for c in merged.columns if "scheme" in c.lower()][0]
else:
    scheme_col = scheme_col[0]

fund_col = [c for c in merged.columns if c == "fund_house"]
if len(fund_col) == 0:
    fund_col = [c for c in merged.columns if "fund_house" in c.lower()][0]
else:
    fund_col = fund_col[0]

print("  Using scheme column:", scheme_col, "and fund column:", fund_col)

# Rank each metric (handle NaN by filtering valid funds only)
valid_mask = merged["return_3yr_pct"].notna() & merged["sharpe_ratio"].notna() & merged["alpha"].notna()
valid = merged[valid_mask].copy()

if len(valid) > 0:
    # Rank metrics - lower expense ratio and max drawdown are better
    valid["return_3yr_rank"] = valid["return_3yr_pct"].rank(ascending=False, method="min")
    valid["sharpe_rank"] = valid["sharpe_ratio"].rank(ascending=False, method="min")
    valid["alpha_rank"] = valid["alpha"].rank(ascending=False, method="min")
    valid["expense_rank"] = valid["expense_ratio_pct"].rank(ascending=True, method="min")  # Lower is better
    valid["max_dd_rank"] = valid["max_drawdown_pct"].rank(ascending=True, method="min")  # Less negative is better
    
    # Composite score: 30% return + 25% sharpe + 20% alpha + 15% expense (inverse) + 10% max DD (inverse)
    # Using 41-rank transform since we have 40 funds (maps rank 1->40 to 40->1)
    valid["composite_score"] = (
        0.30 * valid["return_3yr_rank"] +
        0.25 * valid["sharpe_rank"] +
        0.20 * valid["alpha_rank"] +
        0.15 * (41 - valid["expense_rank"]) +  # Inverse: lower expense = better rank
        0.10 * (41 - valid["max_dd_rank"])     # Less negative DD = better rank
    )
    valid["composite_score"] = valid["composite_score"].clip(upper=100).round(2)
    
    # Map composite scores back to ALL funds
    all_codes = merged["amfi_code"].unique()
    code_to_score = dict(zip(valid["amfi_code"], valid["composite_score"]))
    merged["composite_score"] = [code_to_score.get(c, np.nan) for c in merged["amfi_code"]]
else:
    merged["composite_score"] = np.nan

# Sort by composite score (descending) - put NaN at bottom
merged = merged.sort_values("composite_score", ascending=False, na_position="last")

# Build fund scorecard - select columns including the resolved scheme/fund house names
fund_scorecard = merged[[
    "amfi_code", scheme_col, fund_col, 
    "return_3yr_pct", "sharpe_ratio", "alpha", "beta",
    "max_drawdown_pct", "expense_ratio_pct", "composite_score"
]].copy()

fund_scorecard.to_csv(os.path.join(PROCESSED_DIR, "fund_scorecard.csv"), index=False)
print("  [OK] Fund scorecard saved: Top funds by composite score")
print("  Top 5 funds by composite score:")
top5 = fund_scorecard.head(5)
for _, row in top5.iterrows():
    print("  " + str(row[scheme_col]) + " (" + str(row[fund_col]) + "): " + str(row["composite_score"]))

### Task 8: Benchmark comparison chart - Plot top 5 funds vs Nifty 50 and Nifty 100

In [ ]:
print("\n[Task 8] Benchmark comparison chart...")

# Get top 5 funds from scorecard
top5_codes = fund_scorecard["amfi_code"].head(5).tolist()

# Calculate NAV growth for top 5 funds vs benchmarks over 3 years
bench_indices = ["Nifty 50", "Nifty 100"]
fig = go.Figure()

# Add benchmark traces
for idx_name in bench_indices:
    idx_data = bench_raw[bench_raw["index_name"] == idx_name].sort_values("date")
    if len(idx_data) > 0:
        # Normalize to 100 at start
        first_val = idx_data["close_value"].iloc[0]
        normalized = (idx_data["close_value"] / first_val) * 100
        fig.add_trace(go.Scatter(
            x=idx_data["date"],
            y=normalized,
            name=idx_name,
            line=dict(width=2)
        ))

# Add top 5 fund NAV traces
for code in top5_codes:
    fund_nav = nav_clean[nav_clean["amfi_code"] == code].sort_values("date")
    if len(fund_nav) > 0:
        first_nav = fund_nav["nav"].iloc[0]
        normalized_nav = (fund_nav["nav"] / first_nav) * 100
        fig.add_trace(go.Scatter(
            x=fund_nav["date"],
            y=normalized_nav,
            name="Fund " + str(code),
            line=dict(width=1, dash="dot")
        ))

fig.update_layout(
    title="Top 5 Funds vs Benchmark Indices (3-Year Normalized Growth, Base=100)",
    xaxis_title="Date",
    yaxis_title="Normalized Value (Base 100)",
    hovermode="x",
    height=600,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig.write_html(os.path.join("outputs", "benchmark_comparison.html"))
print("  [OK] Benchmark comparison chart saved")

### Summary

In [ ]:
print("\n" + "=" * 60)
print("DAY 4 COMPLETE: Fund Performance Analytics")
print("=" * 60)
print("\nMetrics computed and saved:")
print("  - daily_returns.csv: Daily returns for all funds")
print("  - cagr_report.csv: 1yr/3yr/5yr CAGR per fund")
print("  - sharpe_report.csv: Sharpe Ratio per fund (Rf=6.5%)")
print("  - sortino_report.csv: Sortino Ratio per fund")
print("  - alpha_beta_report.csv: Alpha & Beta from original data")
print("  - max_drawdown_report.csv: Maximum Drawdown per fund")
print("  - fund_scorecard.csv: Composite score 0-100 ranking")
print("  - benchmark_comparison.html: Top 5 funds vs Nifty 50/100")
print("\nOutput files in outputs/ directory:")
import os
for f in os.listdir("outputs"):
    if f.endswith(".csv") or f.endswith(".html"):
        print("  - outputs/" + f)
print("=" * 60)